# Lab 02 — Housing Price Regression — Full Workflow
**Regression Track** · Intermediate · ~75 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Preprocess mixed numeric/categorical features
2. Compare LinearRegression vs RandomForest
3. Use cross-validation for honest evaluation
4. Read feature importance for model insight

## Datasets (this folder)
- `housing.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-02-housing-price-regression/lab-02-housing-price-regression.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`housing.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-02-housing-price-regression"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-02-housing-price-regression/bundle/dataset.zip"
NEED = ["housing.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Real Estate: Predicting California Housing Prices

> **Scenario:** You are a data science consultant for a real estate investment firm. The firm wants to predict median house values in California districts using features like location, income, and housing density. You have 20,640 census block-group records. Your task is to build and compare regression models, handle missing data, encode categorical variables, and identify the most important features.
>
> **You will learn:** train/test split, median imputation, one-hot encoding, LinearRegression vs RandomForest, cross-validation, feature importance.
> **Time:** ~75 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | What it means | Where it shows up |
|---|---|---|
| Train/test split | Hold out 20% of data for unbiased evaluation | `train_test_split(test_size=0.2)` |
| Median imputation | Fill missing values with the column median | `SimpleImputer(strategy="median")` |
| One-hot encoding | Convert categories to binary columns | `pd.get_dummies()` |
| LinearRegression | Linear model — fast, interpretable | `LinearRegression()` |
| RandomForest | Ensemble of decision trees — captures non-linearity | `RandomForestRegressor()` |
| Cross-validation | K-fold evaluation for robust estimates | `cross_val_score()` |
| Feature importance | Which features drive predictions | `rf.feature_importances_` |

---

## 1. Load and Explore the Data

The California housing dataset contains 20,640 census block groups. The target is `median_house_value` (in hundreds of thousands of dollars). Features include location (longitude/latitude), housing age, room counts, population, households, median income, and ocean proximity category.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\housing.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())
print(df["ocean_proximity"].value_counts())


**What to notice:**
- 20,640 rows, 10 columns (9 features + 1 target).
- `total_bedrooms` has missing values — we must impute them.
- `ocean_proximity` is categorical with 5 distinct values — needs encoding.
- `median_house_value` ranges up to 500,001 (capped).

> **Pitfall:** Never fit an imputer on the full dataset before splitting — that leaks test information into training. Fit on train only.

---

## 2. Preprocess: Impute and Encode

We split first, then fit a median imputer on the training data only. For the categorical `ocean_proximity` column, we use one-hot encoding so both linear and tree models can consume it.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_test_num = pd.DataFrame(
    imputer.transform(X_test.select_dtypes(include=[np.number])),
    columns=X_test.select_dtypes(include=[np.number]).columns,
    index=X_test.index,
)

X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_test_cat = pd.get_dummies(X_test[["ocean_proximity"]], drop_first=True)
X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)

X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_proc = pd.concat([X_test_num, X_test_cat], axis=1)

print(f"Train shape: {X_train_proc.shape}")
print(f"Test shape:  {X_test_proc.shape}")
print(f"Missing in train: {X_train_proc.isnull().sum().sum()}")
print(f"Missing in test:  {X_test_proc.isnull().sum().sum()}")


**What to notice:**
- After imputation, zero missing values remain.
- One-hot encoding adds 4 binary columns (5 categories − 1 dropped reference).
- The processed feature matrix has 13 columns (9 numeric + 4 one-hot).

> **Pitfall:** `pd.get_dummies` on train and test separately can produce different columns. Use `reindex` to align test columns with train.

---

## 3. Baseline: Mean Predictor

Before building a real model, establish a baseline. The simplest possible model always predicts the training mean. Any real model should beat this.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline_pred = np.full_like(y_test, y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline (mean) MAE: ${baseline_mae:,.2f}")


**What to notice:**
- The baseline MAE tells you how far off a naive guess would be.
- This is the number your models must beat to be useful.

> **Pitfall:** Always compute the baseline on the training mean, not the test mean — using test data for the baseline is leakage.

---

## 4. Linear Regression vs Random Forest

We fit both a LinearRegression and a RandomForestRegressor on the same preprocessed data and compare their test MAE.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_test_num = pd.DataFrame(
    imputer.transform(X_test.select_dtypes(include=[np.number])),
    columns=X_test.select_dtypes(include=[np.number]).columns,
    index=X_test.index,
)
X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_test_cat = pd.get_dummies(X_test[["ocean_proximity"]], drop_first=True)
X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)
X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_proc = pd.concat([X_test_num, X_test_cat], axis=1)

lr = LinearRegression()
lr.fit(X_train_proc, y_train)
lr_pred = lr.predict(X_test_proc)
lr_mae = mean_absolute_error(y_test, lr_pred)
print(f"Linear Regression MAE: ${lr_mae:,.2f}")

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_proc, y_train)
rf_pred = rf.predict(X_test_proc)
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f"Random Forest MAE:     ${rf_mae:,.2f}")
print(f"Improvement:           ${lr_mae - rf_mae:,.2f} ({(1 - rf_mae/lr_mae)*100:.1f}%)")


**What to notice:**
- Random Forest typically outperforms Linear Regression on this dataset because it captures non-linear relationships and feature interactions.
- The improvement in MAE shows how much those non-linear patterns matter.

> **Pitfall:** Random Forests can overfit if trees are too deep. Use `max_depth` or `min_samples_leaf` to control complexity.

---

## 5. Cross-Validation Comparison

A single train/test split can be noisy. We use 5-fold cross-validation to get a more robust estimate of model performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)

lr_cv = cross_val_score(
    LinearRegression(), X_train_proc, y_train,
    cv=5, scoring="neg_mean_absolute_error"
)
rf_cv = cross_val_score(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_proc, y_train, cv=5, scoring="neg_mean_absolute_error"
)

print(f"LR  CV MAE: ${-lr_cv.mean():,.2f} (+/- ${lr_cv.std():,.2f})")
print(f"RF  CV MAE: ${-rf_cv.mean():,.2f} (+/- ${rf_cv.std():,.2f})")


**What to notice:**
- Cross-validation gives a mean and standard deviation across 5 folds.
- If RF's CV MAE is lower than LR's, the improvement is robust, not a fluke of one split.

> **Pitfall:** Use `neg_mean_absolute_error` (sklearn convention: higher is better, so MAE is negated). Remember to negate the result.

---

## 6. Feature Importance

Random Forest provides feature importances that show which features drive predictions. We plot them to identify the top predictors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_proc, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train_proc.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance")
ax.set_title("Random Forest — Feature Importance (Housing)")
plt.tight_layout()
plt.show()

print("Top 3 features:")
print(importances.sort_values(ascending=False).head(3))


**What to notice:**
- `median_income` is almost always the dominant predictor.
- `ocean_proximity` one-hot columns and location features (longitude/latitude) also contribute.
- Feature importance helps you explain the model to stakeholders.

> **Pitfall:** Feature importance from Random Forest can be biased toward high-cardinality features. Use permutation importance for a more reliable ranking.

---

## Exercises (do these!)

### Exercise 1 — Baseline Mean-Predictor MAE
Compute the MAE of a model that always predicts the training-set mean house value.
*Expected: $92,893.94*

**Follow-up:** Why is this baseline important? Check: baseline MAE value.

### Exercise 2 — Random Forest vs Linear Regression MAE
Compare the test MAE of both models. Which is better and by how much?
*Expected: RF MAE = $31,889.38; LR MAE = $50,752.61*

**Follow-up:** Why does Random Forest outperform Linear Regression here? Check: improvement percentage.

### Exercise 3 — Top 3 Features
Identify the three most important features from the Random Forest.
*Expected: median_income (0.4839), ocean_proximity_INLAND (0.1465), longitude (0.1073).*

**Follow-up:** What does `median_income` dominating tell you about the housing market? Check: top feature name and importance value.

<details>
<summary>Hint</summary>

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_test_num = pd.DataFrame(
    imputer.transform(X_test.select_dtypes(include=[np.number])),
    columns=X_test.select_dtypes(include=[np.number]).columns,
    index=X_test.index,
)
X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_test_cat = pd.get_dummies(X_test[["ocean_proximity"]], drop_first=True)
X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)
X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_proc = pd.concat([X_test_num, X_test_cat], axis=1)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_proc, y_train)

# Exercise 1
baseline_pred = np.full_like(y_test, y_train.mean())
print(mean_absolute_error(y_test, baseline_pred))

# Exercise 2
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train_proc, y_train)
print(f"LR MAE: {mean_absolute_error(y_test, lr.predict(X_test_proc)):.2f}")
print(f"RF MAE: {mean_absolute_error(y_test, rf.predict(X_test_proc)):.2f}")

# Exercise 3
importances = pd.Series(rf.feature_importances_, index=X_train_proc.columns)
print(importances.sort_values(ascending=False).head(3))
```

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_test_num = pd.DataFrame(
    imputer.transform(X_test.select_dtypes(include=[np.number])),
    columns=X_test.select_dtypes(include=[np.number]).columns,
    index=X_test.index,
)
X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_test_cat = pd.get_dummies(X_test[["ocean_proximity"]], drop_first=True)
X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)
X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_proc = pd.concat([X_test_num, X_test_cat], axis=1)

lr = LinearRegression()
lr.fit(X_train_proc, y_train)
lr_mae = mean_absolute_error(y_test, lr.predict(X_test_proc))

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_proc, y_train)
rf_mae = mean_absolute_error(y_test, rf.predict(X_test_proc))

# --- Solution 1 ---
baseline_pred = np.full_like(y_test, y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline MAE: ${baseline_mae:,.2f}")

# --- Solution 2 ---
print(f"LR MAE: ${lr_mae:,.2f}")
print(f"RF MAE: ${rf_mae:,.2f}")
print(f"RF is ${(lr_mae - rf_mae):,.2f} better ({(1 - rf_mae/lr_mae)*100:.1f}% improvement)")

# --- Solution 3 ---
importances = pd.Series(rf.feature_importances_, index=X_train_proc.columns)
top3 = importances.sort_values(ascending=False).head(3)
print("Top 3 features:")
print(top3)

# --- Follow-up 1 ---
print(f"Baseline MAE = ${baseline_mae:,.2f} — any useful model must beat this.")

# --- Follow-up 2 ---
assert rf_mae < lr_mae, "Random Forest should beat Linear Regression"
print(f"RF beats LR by ${(lr_mae - rf_mae):,.2f}")

# --- Follow-up 3 ---
assert top3.index[0] == "median_income", "median_income should be the top feature"
print(f"Top feature: {top3.index[0]} (importance = {top3.iloc[0]:.4f})")


### What to learn next
- Gradient boosting (XGBoost, LightGBM) for even better regression performance
- Hyperparameter tuning with GridSearchCV or RandomizedSearchCV
- Feature engineering: creating interaction terms and polynomial features

*Files in this folder: housing.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
